# OmniRoute on Google Colab

Installs [OmniRoute](https://www.npmjs.com/package/omniroute) (a unified AI router with an OpenAI-compatible API), starts the server and opens a public URL for the dashboard.

Run the cells from top to bottom. The server only lives as long as this Colab runtime.

## 1. Install Node.js 24
OmniRoute needs Node 22.22.2+ or 24+, and the Node that ships with Colab is older.

In [ ]:
!curl -fsSL https://deb.nodesource.com/setup_24.x | sudo -E bash - > /dev/null
!sudo apt-get install -y nodejs > /dev/null
!node -v && npm -v

## 2. Install OmniRoute
This takes about 3–5 minutes.

In [ ]:
!npm install -g omniroute --loglevel=error
!omniroute --version

## 3. Start the server in the background

In [ ]:
import subprocess, time, urllib.request

PORT = 20128
log = open("/content/omniroute.log", "w")
server = subprocess.Popen(["omniroute", "serve"], stdout=log, stderr=subprocess.STDOUT)

for _ in range(60):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}", timeout=2)
        print(f"OmniRoute is running on port {PORT}")
        break
    except Exception:
        time.sleep(2)
else:
    print("Server did not start. Last log lines:")
    print(open("/content/omniroute.log").read()[-3000:])

In [ ]:
!omniroute health

## 4. Open the dashboard via a public URL
Uses a free Cloudflare quick tunnel. **Anyone with this URL can reach your server**, so set a dashboard password in OmniRoute and don't share the link.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import re, subprocess, time
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0)
        break

print("Dashboard:", public_url)
print("API base: ", f"{public_url}/v1")

## 5. Try the OpenAI-compatible API
First add a provider (API key or OAuth) in the dashboard, then create an API key there and paste it below.

In [ ]:
import requests

OMNIROUTE_API_KEY = ""  # paste a key from the OmniRoute dashboard
BASE = f"http://localhost:{PORT}/v1"
headers = {"Authorization": f"Bearer {OMNIROUTE_API_KEY}"} if OMNIROUTE_API_KEY else {}

models = requests.get(f"{BASE}/models", headers=headers).json()
print([m["id"] for m in models.get("data", [])][:20])

In [ ]:
MODEL = ""  # pick one of the ids printed above

r = requests.post(f"{BASE}/chat/completions", headers=headers, json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Say hello in one line."}],
})
print(r.json())

## 6. Stop everything

In [ ]:
tunnel.terminate()
server.terminate()
!omniroute stop || true
print("Stopped")